<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); border-radius: 16px; padding: 40px; margin-bottom: 24px;">
  <div style="display: flex; align-items: center; gap: 24px;">
    <img src="logo.png" alt="Gradients" style="height: 80px;">
    <div>
      <h1 style="color: #fff; margin: 0; font-size: 2em;">Fine-Tune Any Model in 3 Steps</h1>
      <p style="color: #a8b2d1; margin: 8px 0 0 0; font-size: 1.1em;">Train a medical QA model on PubMedQA, then compare it against the base model — no infrastructure, no MLOps, just an API key.</p>
    </div>
  </div>
</div>

## 1. Install

In [ ]:
%pip install -q --upgrade gradientsio==0.1.2

## 2. Train

Point at a model and a dataset. That's it.

In [ ]:
import os, gradientsio as gradients

os.environ["GRADIENTS_API_KEY"] = "paste-your-api-key-here"
client = gradients.GradientsClient()

trained_model = client.train(
    model="Qwen/Qwen2.5-3B",
    task_type=gradients.TaskType.INSTRUCT,
    hours=2,
    dataset="gradients-io-tournaments/PubMedQA-Normalized-Train",
    field_instruction="instruction",
    field_input="input",
    field_output="output",
).wait().trained_model_repository

## 3. Compare

Load held-out test questions the model has never seen, run both base and trained, and see the difference.

In [ ]:
from IPython.display import HTML, display

samples = gradients.load_dataset_rows("gradients-io-tournaments/PubMedQA-Normalized-Test")
prompts = [f"{r['instruction']}\n\nAnswer:" for r in samples]

sampler = gradients.ModelSampler()
base_answers = sampler.generate("Qwen/Qwen2.5-3B", prompts)
trained_answers = sampler.generate_with_adapter(trained_model, prompts, base_model_repo="Qwen/Qwen2.5-3B")

# --- styled comparison cards ---
CARD_CSS = """
<style>
.grad-card { border: 1px solid #e2e8f0; border-radius: 12px; margin: 16px 0; overflow: hidden; font-family: system-ui, -apple-system, sans-serif; }
.grad-card-header { background: linear-gradient(135deg, #1a1a2e, #0f3460); color: #fff; padding: 16px 20px; font-weight: 600; }
.grad-columns { display: grid; grid-template-columns: 1fr 1fr 1fr; }
.grad-col { padding: 16px 20px; border-right: 1px solid #e2e8f0; }
.grad-col:last-child { border-right: none; }
.grad-col-label { font-size: 0.75em; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 8px; font-weight: 700; }
.grad-expected .grad-col-label { color: #6b7280; }
.grad-trained .grad-col-label { color: #7c3aed; }
.grad-trained { background: #f5f3ff; }
.grad-base .grad-col-label { color: #9ca3af; }
.grad-base { color: #6b7280; }
.grad-col p { margin: 0; line-height: 1.6; font-size: 0.92em; }
</style>
"""

cards = [CARD_CSS]
for i, (row, base, trained) in enumerate(zip(samples, base_answers, trained_answers), 1):
    cards.append(f"""
<div class="grad-card">
  <div class="grad-card-header">Q{i}: {row['instruction'][:200]}</div>
  <div class="grad-columns">
    <div class="grad-col grad-expected"><div class="grad-col-label">Expected</div><p>{row.get('output', '').strip()}</p></div>
    <div class="grad-col grad-trained"><div class="grad-col-label">Trained Model</div><p>{trained.strip()}</p></div>
    <div class="grad-col grad-base"><div class="grad-col-label">Base Model</div><p>{base.strip()}</p></div>
  </div>
</div>""")

display(HTML("\n".join(cards)))